# 図の入力：元画像・縮小画像・PDFを明示する

授業時間は[演習カード](../activities/day1.md)を確認します。質問欄は空欄から始めます。
送信セルで `SEND=True` にした時だけ有料APIを呼びます。モデル・接続経路・許可枠を明示してください。APIキーは入力欄でのみ受け取り、保存しません。
送信失敗時は停止します。別モデルへの切替・再送は自動で行いません。大学GUIの記録とAPIの記録は経路を区別します。

**スライドとの対応**：55「図2をPNGで読み、自分の説明と比べる」、64–74「画像と言語・PDF」、75「元画像と縮小画像を、同じ質問で比べる」。
元画像と縮小画像では、質問・モデル・出力上限を同じにし、両方を新規会話で試します。元画像の正しい回答が縮小画像の履歴へ入らないようにします。


In [ ]:
# @title 0. 教材を準備する { display-mode: "form" }
from pathlib import Path
import sys
PREPARE_COLAB = False  # @param {type:"boolean"}
RELEASE_COMMIT = "3730accec6aee1870bd5763d072474c0b6d5afd3"
RELEASE_SHA256 = "47b0f18d47fe6ee210712c24a7c4bcc95532b9e4c273f07c81e3d41f9abc87a3"
BOOTSTRAP_SHA256 = "f1ad167aeec1cf95e2aa835eef73c05fdeae916445985ab69679ec6060d99914"
if PREPARE_COLAB:
    from urllib.request import urlopen
    from hashlib import sha256
    if not RELEASE_COMMIT or not RELEASE_SHA256:
        raise RuntimeError('この改訂版のColab公開準備はまだ完了していません。')
    url = f'https://raw.githubusercontent.com/humansys-lab/multimodal-ai-paper-reading/{RELEASE_COMMIT}/scripts/colab_bootstrap.py'
    with urlopen(url, timeout=40) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or sha256(source).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError('準備コードのハッシュが一致しません。実行を中止しました。')
    setup = {}
    exec(compile(source, 'verified_colab_bootstrap.py', 'exec'), setup)
    ROOT = setup['prepare_notebook'](RELEASE_COMMIT, RELEASE_SHA256, profile='documents')
else:
    ROOT = globals().get('ROOT', Path.cwd()).resolve()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材フォルダで実行するか、ColabではPREPARE_COLABをTrueにしてください。')
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
print('教材フォルダ:', ROOT.name)

## 1. 対話を準備する

In [ ]:
from copy import deepcopy
from seminar_lab.config import load_yaml, material_context
from seminar_lab.client import Session, OpenAITransport, output_text
from seminar_lab.connection import load_connection
from seminar_lab.ui import activity_description, preview_text, save_pair
from seminar_lab.records import load_records
course = load_yaml(ROOT / 'config/course.yaml')
manifest = load_yaml(ROOT / 'materials/manifest.yaml')

## 2. 教員の接続ファイルを指定する（初回・活動変更時）
教員から配布されたファイルをColab左側のFiles欄へ置き、パスをコピーして`connection_path`へ記入します。学生がモデルの対応表や利用許可を記入する必要はありません。公開の空様式では送信できません。

`V1`＝元画像の読解、`V2`＝縮小画像との比較。共通読解の論文・版・指定範囲・3問は全員同じです。

設定を読み直しても過去の記録と使用回数は保持します。モデルや資料を変えるときは保存してから新規会話にします。金額の利用上限は教員の中継側で管理します。


In [ ]:
from uuid import uuid4
connection_path = ''  # 教員が配布した接続ファイルのパス。キーは書かない
activity_id = 'V1'  # この上の表にある演習名と対応
if 'session' not in globals():
    session = Session()
    permissions = {}  # 同じ許可の再読込みで使用回数をリセットしない
if 'run_id' not in globals():
    run_id = str(uuid4())
runtime = permission = None
if connection_path.strip():
    runtime, permission = load_connection(connection_path, permissions)
    print('モデル:', runtime['model'], '／ 接続経路:', runtime['route'])
else:
    print('接続は未設定です。教員の設定ファイルを指定すると送信を準備できます。')
print(activity_description(course, manifest, activity_id))


## 3. 自分の質問を書く
複数行の質問や抜粋は、`"""`と`"""`の間に書けます。引用符そのものを含む文章は、囲み方に注意してください。


In [ ]:
question = ''
conversation_mode = 'new'  # 比較では毎回new。前の画像を履歴に含めない
parameters = {'max_output_tokens': 512}

## 4. 送る画像かPDFを1つ準備する
Colabのファイル欄へファイルをアップロードし、パスをコピーします。`PREPARE_INPUT=True`で実行。最初は`image`、元画像の結果を保存したら`resized_image`へ切り替えます。表示する画像は実際の送信画像です。

PDFは発展：`pdf`を選び、送信ページと原本ページの対応を確認します。PDF内部の処理はサービス側で行われます。画像読取に失敗してもテキスト抽出へ自動変更しません。
**見る場所**：同じ文字・軸ラベル・数値を原図と回答で照合します。回答の流暢さだけでは読めたと判断しません。PDFは、サービスが本文とページ画像を扱う方式です。こちらで本文だけへ置換しません。[OpenAI公式のPDF入力](https://developers.openai.com/api/docs/guides/file-inputs)


In [ ]:
PREPARE_INPUT = False
input_kind = 'image'  # 'image' / 'resized_image' / 'pdf'から1つ
local_path = ''
source_location = ''  # 元の論文・ページ・図
width = 600  # resized_imageだけで使う
pdf_pages = []  # pdfだけで使う。物理ページを1始まりで指定（例：[1, 3]）
inputs = []
if PREPARE_INPUT:
    if input_kind in {'image', 'resized_image'}:
        from seminar_lab.inputs import image_input, resized_image_input
        from IPython.display import display, Image
        from base64 import b64decode
        item = (image_input(local_path, source_location) if input_kind == 'image'
                else resized_image_input(local_path, source_location, width))
        display(Image(data=b64decode(item.content['image_url'].split(',', 1)[1]), width=700))
        print('画面では幅700pxで表示。送信する画素数は下のpixelsで確認します。')
    elif input_kind == 'pdf':
        from seminar_lab.inputs import pdf_input
        item = pdf_input(local_path, pdf_pages)
    else:
        raise ValueError('入力方式はimage / resized_image / pdfを指定してください。')
    print('実際に送る入力:', item.provenance)
    inputs = [item]  # 準備が成功した入力だけを使う

## 5. 質問・全履歴・入力をプレビューする（無料）

In [ ]:
previewed_request = None
if question.strip():
    if runtime is None or permission is None:
        raise ValueError('教員の接続ファイルを先に読み込んでください。')
    if not inputs:
        raise ValueError('画像またはPDFを準備してください。')
    context = (globals().get('homework_context') if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    if context is None:
        raise ValueError('Homeworkの論文情報を01の末尾にある切替セルで確認してください。')
    payload = session.preview(runtime, question, inputs, conversation_mode, parameters)
    previewed_request = deepcopy((context, payload, [x.provenance for x in inputs]))
    print('資料と版・指定範囲:', context['material_id'], context.get('source_version'), context.get('assigned_scope'))
    print('実入力の由来:', [x.provenance for x in inputs])
    print(preview_text(payload))
else:
    print('質問欄は空です。自分の問いを入力してください。送信していません。')

## 6. 確認した内容を1回送る（ここで課金）
`SEND=True`で実行し、専用入力欄へ個別キーを入れます。同じ実行番号の再送は停止します。回答が途中・拒否・エラーならその状態を表示し、別モデルへの切替や再送はしません。

In [ ]:
SEND = False
if SEND:
    result = None  # 失敗後に前の回答を今回の結果として扱わない
    if not inputs:
        raise ValueError('画像またはPDFを準備してください。')
    if runtime is None or permission is None:
        raise ValueError('教員の接続ファイルを先に読み込んでください。')
    context = (globals().get('homework_context') if activity_id == 'HW1' else material_context(course, manifest, activity_id))
    if context is None:
        raise ValueError('Homeworkの論文情報を01の末尾にある切替セルで確認してください。')
    # キー入力より先に、素材・設定・送信内容を検証
    payload = session.preview(runtime, question, inputs, conversation_mode, parameters)
    if globals().get('previewed_request') != (context, payload, [x.provenance for x in inputs]):
        raise ValueError('質問・入力・活動・設定が変わっています。プレビューを再実行してください。')
    permission.validate(runtime)
    session.check_run_id(run_id)
    from getpass import getpass
    transport = OpenAITransport(runtime, getpass('許可された個別APIキー（表示・保存しません）: '), permission)
    try:
        result = session.run(runtime=runtime, context=context, prompt=question, inputs=inputs,
                             mode=conversation_mode, parameters=parameters, run_id=run_id, transport=transport)
    finally:
        transport.close()
    print('状態:', result['status'])
    if result.get('error_message'):
        print(result['error_message'])
    print(output_text(result))
    print('費用:', result['cost_value'] if result['cost_value'] is not None else '不明')
else:
    print('送信OFF。APIは呼び出していません。')

## 7. 自分の説明を追記し、保存する
`None` は前の記入を保持。JSONLとMarkdownをダウンロードします。

In [ ]:
# Noneは前の記入を保持。文字列を入れた欄だけ更新する。
annotations = {
    'student_explanation': None,  # 自分の説明
    'evidence_location': None,  # 根拠のページ・図・本文
    'student_revision': None,  # AIの回答を確認して直した説明
    'unresolved_point': None,  # まだ分からないこと
    'next_change': None,  # 次に変える質問や入力
    'change_reason': None,  # 変更する理由
}
SAVE = False
if SAVE:
    if not session.records:
        raise ValueError('保存する実行記録がありません。先に対話を実行してください。')
    if (globals().get('result') is None or result.get('run_id') != run_id
            or session.records[-1]['run_id'] != run_id):
        raise ValueError('今回の実行記録がありません。前回の回答へ今回の説明を付けずに停止しました。')
    if any(value is not None and not isinstance(value, str) for value in annotations.values()):
        raise ValueError('記入欄は文字列かNoneにしてください。')
    session.records[-1].update({key: value for key, value in annotations.items() if value is not None})
    paths = save_pair(session.records, ROOT / 'outputs')
    print([p.name for p in paths])
    assert load_records(paths[0]) == session.records
    assert load_records(paths[1]) == session.records
    # Colabのファイル欄からoutputs内の2ファイルをダウンロード。
else:
    print('保存は未実行です。前の記入内容も変更していません。')

## 次の質問
新しい質問の時だけ新しい実行番号を作ります。同じ送信セルの再実行による二重送信は拒否します。

In [ ]:
START_NEXT_RUN = False
if START_NEXT_RUN:
    run_id = str(uuid4())
    print('新しい実行番号を用意しました。質問・新規/継続・プレビューを見直してください。')